# 10 — Model Registry & Batch Inference

**Owner:** Member 2 — Narendra Iyer 
**Project:** AeroDelay AI — Airline Delay Risk Prediction 
**Course:** AAI-540, Group 3

## Purpose
Register the selected XGBoost model in SageMaker Model Registry with full
metadata, then run Batch Transform on the production simulation dataset
to generate delay-risk predictions for all future-window flights.

This notebook completes Member 2's scope and hands off batch output to
Member 3 (MLOps) for deployment and monitoring work.

## Prerequisites
- `09_model_evaluation.ipynb` completed — `%store` variable `best_model_uri` must be present.

## Notes on Environment Fixes
- `sagemaker.core.helper.session_helper` requires the `sagemaker_core` symlink fix applied each session.
- `XGBoostModel` from `sagemaker.xgboost.model` is used for Batch Transform instead of `sm.create_transform_job`
  because the raw boto3 approach causes a ping endpoint failure.
- A clean model tar.gz (xgboost-model only, without feature_cols.joblib) is required for the serving container.

## 1. Setup

In [1]:
import boto3, os, json, time, tarfile, io
import pandas as pd
import awswrangler as wr
import sagemaker
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess   = Session()
bucket = sess.default_bucket()
role   = get_execution_role()
region = sess.boto_region_name
sm     = boto3.client("sagemaker")
s3     = boto3.client("s3")

# Standard sagemaker session required for XGBoostModel and transformers
# sagemaker_core Session does not have all attributes needed by the SDK
standard_sess = sagemaker.Session()
standard_role = sagemaker.get_execution_role()

print(f"Bucket : {bucket}")
print(f"Region : {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Bucket : sagemaker-us-east-1-151132426745
Region : us-east-1


In [2]:
%store -r s3_aerodelay
%store -r best_model_uri
%store -r best_hp
%store -r xgb_initial_job
%store -r hpo_job_name
%store -r best_hpo_job_name
%store -r baseline_job_name

print(f"s3_aerodelay   : {s3_aerodelay}")
print(f"best_model_uri : {best_model_uri}")

# Check if batch input CSV already exists in S3 from a previous run
resp = s3.list_objects_v2(Bucket=bucket, Prefix="airline-delay/batch-input/")
for obj in resp.get("Contents", []):
    print(obj["Key"], obj["Size"])

s3_aerodelay   : s3://sagemaker-us-east-1-151132426745/airline-delay
best_model_uri : s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/hpo/aerodelay-tuning-260613-1814-001-9a5121f3/output/model.tar.gz
airline-delay/batch-input/production_features.csv 94353731


## 2. Create Model Package Group in Model Registry

A Model Package Group acts as a versioned container for all model candidates.
We register each new model as a version inside this group.

In [3]:
model_package_group_name = "aerodelay-xgboost-delay-prediction"

try:
    sm.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription=(
            "AeroDelay AI — XGBoost binary classifier for flight arrival delay (>=15 min). "
            "AAI-540 Group 3 project."
        ),
    )
    print(f"Created Model Package Group: {model_package_group_name}")
except sm.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Model Package Group already exists: {model_package_group_name}")
    else:
        raise

Model Package Group already exists: aerodelay-xgboost-delay-prediction


## 3. Register Best XGBoost Model as a Version

In [4]:
# Get the XGBoost container image URI for this region
from sagemaker.core.image_uris import retrieve

xgb_image_uri = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)
print(f"XGBoost image URI: {xgb_image_uri}")

XGBoost image URI: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [5]:
# Load evaluation metrics to embed in the registry entry
eval_report_df = wr.s3.read_json(f"{s3_aerodelay}/reports/evaluation_report.json")
xgb_metrics    = eval_report_df["xgboost_best_hpo"].iloc[0]
print("Registering with metrics:", xgb_metrics)

2026-06-13 22:56:39,913	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 408924160 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=0.68gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-13 22:56:41,085	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Registering with metrics: {'accuracy': 0.5942000000000001, 'precision': 0.2139, 'recall': 0.6239, 'f1': 0.3186, 'roc_auc': 0.648}


In [6]:
register_response = sm.create_model_package(
    ModelPackageGroupName    = model_package_group_name,
    ModelPackageDescription  = (
        f"XGBoost best HPO run. "
        f"Test Recall={xgb_metrics['recall']}, "
        f"F1={xgb_metrics['f1']}, "
        f"ROC-AUC={xgb_metrics['roc_auc']}. "
        f"Trained on Q1-2024 BTS airline data."
    ),
    InferenceSpecification    = {
        "Containers": [
            {
                "Image"        : xgb_image_uri,
                "ModelDataUrl" : best_model_uri,
            }
        ],
        "SupportedTransformInstanceTypes"  : ["ml.m5.large", "ml.m5.xlarge"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"],
        "SupportedContentTypes"  : ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus = "Approved",    # Auto-approve after our evaluation gate passed
    ModelMetrics = {
        "ModelQuality": {
            "Statistics": {
                "ContentType" : "application/json",
                "S3Uri"       : f"{s3_aerodelay}/reports/evaluation_report.json",
            }
        }
    },
)

model_package_arn = register_response["ModelPackageArn"]
print(f"Registered model package ARN: {model_package_arn}")

Registered model package ARN: arn:aws:sagemaker:us-east-1:151132426745:model-package/aerodelay-xgboost-delay-prediction/5


In [7]:
# Confirm registration
versions = sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending"
)

print(f"All versions in {model_package_group_name}:")
for v in versions["ModelPackageSummaryList"]:
    print(f"  v{v['ModelPackageVersion']} | {v['ModelApprovalStatus']} | {v['CreationTime']}")

All versions in aerodelay-xgboost-delay-prediction:
  v5 | Approved | 2026-06-13 22:56:46.842000+00:00
  v4 | Approved | 2026-06-13 21:36:19.050000+00:00
  v3 | Approved | 2026-06-13 20:22:16.461000+00:00
  v2 | Approved | 2026-06-13 20:01:21.755000+00:00
  v1 | Approved | 2026-06-13 19:30:48.275000+00:00


## 4. Prepare Batch Input Data

Batch Transform requires CSV without the target column.
String columns must be label-encoded since XGBoost requires numeric input.
We check if the CSV already exists in S3 before regenerating it.

In [8]:
from sklearn.preprocessing import LabelEncoder

def encode_strings(df):
    """Label-encode all string/object columns so XGBoost can process them."""
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

# Load production simulation data
prod_df = wr.s3.read_parquet(f"{s3_aerodelay}/production_simulation/data.parquet")
print(f"Production simulation shape: {prod_df.shape}")
print(f"Delay rate in prod data    : {prod_df['arrdel15'].mean():.1%}")

# Save ground-truth labels separately for later monitoring evaluation
prod_labels = prod_df[["arrdel15"]].copy()

# Check if batch input already exists in S3 — skip regeneration if so
resp = s3.list_objects_v2(Bucket=bucket, Prefix="airline-delay/batch-input/")
existing = [obj["Key"] for obj in resp.get("Contents", [])]

batch_input_path = f"{s3_aerodelay}/batch-input/production_features.csv"

if existing:
    print(f"Batch input already exists in S3 — skipping regeneration: {existing}")
else:
    # Drop target and encode strings before sending to Batch Transform
    prod_features = encode_strings(prod_df.drop(columns=["arrdel15"]).fillna(0))
    wr.s3.to_csv(df=prod_features, path=batch_input_path, index=False, header=False)
    print(f"Batch input saved to: {batch_input_path}")

Production simulation shape: (650421, 20)
Delay rate in prod data    : 20.6%
Batch input already exists in S3 — skipping regeneration: ['airline-delay/batch-input/production_features.csv']


## 5. Create Clean Model Tar.gz for Batch Transform

The XGBoost serving container (`sagemaker-xgboost:1.7-1`) expects **only** the `xgboost-model`
binary file inside the tar.gz. Our training script also saved `feature_cols.joblib` alongside it,
which causes the container's `serve_utils.get_loaded_booster()` to return a list instead of a
Booster object, crashing the ping endpoint.

Fix: extract only `xgboost-model` and repackage as a clean tar.gz.

In [9]:
bucket_name = best_model_uri.split("/")[2]
key         = "/".join(best_model_uri.split("/")[3:])

print("Downloading model.tar.gz from S3...")
obj       = s3.get_object(Bucket=bucket_name, Key=key)
tar_bytes = obj["Body"].read()

print("Extracting xgboost-model binary...")
with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:gz") as tar:
    xgb_file  = tar.extractfile("xgboost-model")
    xgb_bytes = xgb_file.read()

print("Repackaging as clean model.tar.gz (xgboost-model only)...")
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w:gz") as new_tar:
    info      = tarfile.TarInfo(name="xgboost-model")
    info.size = len(xgb_bytes)
    new_tar.addfile(info, io.BytesIO(xgb_bytes))

clean_model_key = "airline-delay/model-artifacts/clean/xgboost-model.tar.gz"
s3.put_object(Bucket=bucket_name, Key=clean_model_key, Body=buf.getvalue())
clean_model_uri = f"s3://{bucket_name}/{clean_model_key}"
print(f"Clean model uploaded to: {clean_model_uri}")

Extracting xgboost-model binary...
Repackaging as clean model.tar.gz (xgboost-model only)...


Clean model uploaded to: s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/clean/xgboost-model.tar.gz


## 6. Run Batch Transform on Production Simulation Data

Batch Transform scores the entire production simulation dataset at once —
this is the primary inference pattern for the project (no real-time endpoint needed).

The production simulation data represents the future-window flights that
the model was never trained on (the last 40% of the time-ordered dataset).

Note: We use `XGBoostModel` with `sagemaker.Session()` (not `sagemaker_core` Session)
because the standard SDK properly wires up the serving container and ping endpoint.

In [10]:
from sagemaker.xgboost.model import XGBoostModel

batch_output_uri = f"{s3_aerodelay}/batch-output/"

# Use clean model URI — xgboost-model only, no feature_cols.joblib
xgb_model_obj = XGBoostModel(
    model_data        = clean_model_uri,
    role              = standard_role,
    framework_version = "1.7-1",
    sagemaker_session = standard_sess,
)

transformer = xgb_model_obj.transformer(
    instance_count = 1,
    instance_type  = "ml.m5.xlarge",
    output_path    = batch_output_uri,
    assemble_with  = "Line",
    accept         = "text/csv",
)

transformer.transform(
    data         = f"{s3_aerodelay}/batch-input/",
    content_type = "text/csv",
    split_type   = "Line",
    wait         = True,
    logs         = True,
)

batch_job_name = transformer.latest_transform_job.name
model_name     = xgb_model_obj.name
print(f"Batch Transform completed.")
print(f"Job name   : {batch_job_name}")
print(f"Output at  : {batch_output_uri}")

INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-06-13-22-57-09-447


.

.

.

.

...

.

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-13:23:01:54:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-13:23:01:54:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-13:23:01:54:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  impor

Batch Transform completed.
Job name   : sagemaker-xgboost-2026-06-13-22-57-09-447
Output at  : s3://sagemaker-us-east-1-151132426745/airline-delay/batch-output/


## 7. Inspect Batch Output

In [11]:
# List batch output files
output_files = wr.s3.list_objects(batch_output_uri)
print("Batch output files:")
for f in output_files:
    print(f"  {f}")

Batch output files:
  s3://sagemaker-us-east-1-151132426745/airline-delay/batch-output/production_features.csv.out


In [12]:
# Read predictions (each line is a probability score 0-1)
pred_df = wr.s3.read_csv(
    path=batch_output_uri,
    header=None,
    names=["delay_probability"]
)

pred_df["predicted_delay"]  = (pred_df["delay_probability"] >= 0.5).astype(int)
pred_df["actual_delay"]     = prod_labels["arrdel15"].values

print(f"Total predictions   : {len(pred_df):,}")
print(f"Predicted delay rate: {pred_df['predicted_delay'].mean():.1%}")
print(f"Actual delay rate   : {pred_df['actual_delay'].mean():.1%}")
pred_df.head(10)

Total predictions   : 650,421
Predicted delay rate: 29.1%
Actual delay rate   : 20.6%


,delay_probability,predicted_delay,actual_delay
0,0.455187,0,0
1,0.491046,0,0
2,0.437632,0,0
3,0.490789,0,0
4,0.478821,0,0
5,0.491939,0,0
6,0.502016,1,0
7,0.477347,0,0
8,0.487247,0,1
9,0.394772,0,0


In [13]:
from sklearn.metrics import recall_score, f1_score, roc_auc_score

prod_recall  = recall_score(pred_df["actual_delay"], pred_df["predicted_delay"], zero_division=0)
prod_f1      = f1_score(pred_df["actual_delay"],     pred_df["predicted_delay"], zero_division=0)
prod_auc     = roc_auc_score(pred_df["actual_delay"], pred_df["delay_probability"])

print("=== Production Simulation Batch Results ===")
print(f"  Recall  : {prod_recall:.4f}")
print(f"  F1      : {prod_f1:.4f}")
print(f"  ROC-AUC : {prod_auc:.4f}")

# Save predictions to S3 for Member 3 (monitoring)
wr.s3.to_parquet(
    df=pred_df,
    path=f"{s3_aerodelay}/batch-output/predictions_with_labels.parquet",
)
print(f"\nPredictions with labels saved for Member 3 monitoring work.")

=== Production Simulation Batch Results ===
  Recall  : 0.4189
  F1      : 0.3471
  ROC-AUC : 0.6260

Predictions with labels saved for Member 3 monitoring work.


## 8. Model Registry — Registration Summary

I registered the best XGBoost model from the HPO run into SageMaker Model Registry under the group `aerodelay-xgboost-delay-prediction`. The model was automatically approved based on its demonstrated superiority on the held-out test set — it passed our evaluation gate by beating the baseline on recall, F1, and ROC-AUC.

**Registration Details:**
- Model Package Group: `aerodelay-xgboost-delay-prediction`
- Registered ARN: `arn:aws:sagemaker:us-east-1:151132426745:model-package/aerodelay-xgboost-delay-prediction/5`
- Approval Status: Approved
- Embedded Metrics: Recall 0.624, F1 0.319, ROC-AUC 0.648
- Source: Best HPO trial `aerodelay-tuning-260613-1814-001-9a5121f3`
- Full evaluation report linked from S3: `airline-delay/reports/evaluation_report.json`

The Model Registry provides versioned tracking of all model candidates. Any future retraining or improvement would be registered as a new version in the same group, making it easy to compare, approve, or roll back models over time.

## 9. Batch Transform — Production Simulation Results

I ran Batch Transform to score all 650,421 production simulation flights — these are the flights from late February through March 2024 that the model was never trained on. This represents a realistic production inference scenario where the model encounters genuinely unseen future data.

**Results on Production Simulation Data:**

| Metric | Test Set | Production Simulation | Change |
|---|---|---|---|
| Recall | 0.624 | 0.419 | -0.205 |
| F1 | 0.319 | 0.347 | +0.028 |
| ROC-AUC | 0.648 | 0.626 | -0.022 |
| Predicted delay rate | — | 29.1% | — |
| Actual delay rate | — | 20.6% | — |

**Interpretation:**

The model performs moderately well on unseen production data, though recall drops compared to the test set. There are two reasons for this. First, the production simulation data covers a later time window (late February through March) where delay patterns differ from the training period (January through early February). Second, the predicted delay rate (29.1%) is higher than the actual rate (20.6%), indicating the model over-predicts delays on this future window — a sign of mild distribution shift between training and production periods.

This is expected behavior for any model trained on historical data and deployed on future data. It is not a failure — it is a signal that continual retraining on fresh data would be beneficial in a real production system, which is exactly the kind of monitoring and retraining workflow that notebook 06 (Model Monitor baseline) and Member 3's MLOps work are designed to support.

All predictions are saved to S3 as `predictions_with_labels.parquet` and all key variables are stored via `%store` for Member 3 to use in monitoring and deployment.

## 10. Store Final Variables for Member 3

In [14]:
model_package_group = model_package_group_name
batch_output_s3     = batch_output_uri

%store model_package_group
%store model_package_arn
%store batch_job_name
%store batch_output_s3
%store model_name

print("Stored for Member 3:")
print(f"  model_package_group : {model_package_group}")
print(f"  model_package_arn   : {model_package_arn}")
print(f"  batch_job_name      : {batch_job_name}")
print(f"  batch_output_s3     : {batch_output_s3}")
print(f"  model_name          : {model_name}")

Stored 'model_package_group' (str)
Stored 'model_package_arn' (str)
Stored 'batch_job_name' (str)
Stored 'batch_output_s3' (str)
Stored 'model_name' (str)
Stored for Member 3:
  model_package_group : aerodelay-xgboost-delay-prediction
  model_package_arn   : arn:aws:sagemaker:us-east-1:151132426745:model-package/aerodelay-xgboost-delay-prediction/5
  batch_job_name      : sagemaker-xgboost-2026-06-13-22-57-09-447
  batch_output_s3     : s3://sagemaker-us-east-1-151132426745/airline-delay/batch-output/
  model_name          : sagemaker-xgboost-2026-06-13-22-57-08-742


## 11. Member 2 Completion Summary

| Deliverable | Status | Location |
|---|---|---|
| Logistic Regression baseline | ✓ | `07_baseline_model.ipynb` |
| XGBoost training job | ✓ | `08_xgboost_training.ipynb` |
| HPO (10 trials) | ✓ | `08_xgboost_training.ipynb` |
| Test set evaluation + charts | ✓ | `09_model_evaluation.ipynb` |
| ROC, PR, confusion matrix, feature importance | ✓ | `reports/` folder |
| Subgroup error analysis | ✓ | `reports/subgroup_*.png` |
| Model registered in Model Registry | ✓ | `aerodelay-xgboost-delay-prediction` group |
| Batch Transform on production simulation | ✓ | S3 `batch-output/` |
| Predictions saved for Member 3 monitoring | ✓ | `batch-output/predictions_with_labels.parquet` |